# Inference code for SWIN-Base

## Get modules

In [1]:
import argparse
import os
from time import perf_counter
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from safetensors.torch import load_file
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from dataclasses import dataclass

In [2]:
#import custom Config
from Config import cfg

#device
cfg.device = torch.device(cfg.device)
print(cfg.device)

cuda


## Definitions

In [3]:
def load_model(checkpoint_path: str, precision: str, device: torch.device) -> torch.nn.Module:
    """Load Swin-Base from a .safetensors checkpoint.

    Args:
        checkpoint_path : path to .safetensors file
        precision       : "fp32" or "bf16"
        device          : torch device

    Returns:
        model in eval mode at the requested precision
    """
    print(f"Loading model from: {checkpoint_path}")

    model = timm.create_model(
        cfg.model_name,
        pretrained  = False,   # weights come from the checkpoint
        num_classes = cfg.num_classes,
        drop_rate   = cfg.drop_rate,     # must match training config
        drop_path_rate = cfg.drop_path_rate,  # must match training config
    )

    #load safetensors weights
    #checkpoint is loaded on cpu first and later moved to gpu for compatibility reasons
    state_dict = load_file(checkpoint_path, device = "cpu")
    missing, unexpected = model.load_state_dict(state_dict, strict = True)

    if missing:
        print(f"  Warning — missing keys  : {missing}")
    if unexpected:
        print(f"  Warning — unexpected keys: {unexpected}")

    #cast precision
    if precision == "bf16":
        if not torch.cuda.is_bf16_supported():
            print("  Warning: BF16 not supported on this GPU, falling back to FP32.")
            precision = "fp32"
        else:
            model = model.to(torch.bfloat16)
            print("  Precision: BF16")
    else:
        print("  Precision: FP32")

    model = model.to(device)
    model.eval()

    total_params = sum(p.numel() for p in model.parameters()) / 1e6
    print(f"  Model: {cfg.model_name}  |  {total_params:.1f} M params  |  device: {device}")
    
    return model, precision

In [4]:
def get_inference_transform() -> A.Compose:
    """Validation-equivalent transform — resize + ImageNet normalisation only.
    No augmentation. Must match the val transform used during training.
    """
    return A.Compose([
        
        A.Resize(cfg.img_size, 
                 cfg.img_size),
        
        A.Normalize(mean = cfg.imagenet_mean, 
                    std = cfg.imagenet_std),
        
        ToTensorV2(),
    ])

In [5]:
class ImageDataset(Dataset):
    """Minimal inference dataset — no labels, just images and their paths.

    Skips unreadable files gracefully and records them in self.failed.
    """

    def __init__(self, image_paths: list[str], transform: A.Compose):
        self.transform   = transform
        self.valid_paths = []
        self.failed      = []

        for p in image_paths:
            img = cv2.imread(p)
            if img is None:
                self.failed.append(p)
            else:
                self.valid_paths.append(p)

        if self.failed:
            print(f"  Warning: {len(self.failed)} image(s) could not be read and will be skipped:")
            for f in self.failed[:5]:
                print(f"    {f}")
            if len(self.failed) > 5:
                print(f"    ... and {len(self.failed) - 5} more")

    def __len__(self) -> int:
        return len(self.valid_paths)

    def __getitem__(self, idx: int):
        path  = self.valid_paths[idx]
        image = cv2.imread(path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = self.transform(image=image)["image"]
        return image, path

In [6]:
#define acceptable file types
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tiff"}

def collect_image_paths(input_path: str) -> list[str]:
    """Accept a single image file or a directory (non-recursive)."""
    p = Path(input_path)
    
    if p.is_file():
        if p.suffix.lower() not in IMAGE_EXTENSIONS:
            raise ValueError(f"Unsupported file extension: {p.suffix}")
        return [str(p)]
    
    elif p.is_dir():
        paths = sorted(
            str(f) for f in p.iterdir()
            if f.is_file() and f.suffix.lower() in IMAGE_EXTENSIONS
        )
        if not paths:
            raise FileNotFoundError(f"No images found in directory: {input_path}")
        return paths
    
    else:
        raise FileNotFoundError(f"Input path does not exist: {input_path}")

In [7]:
@torch.no_grad()
def run_inference(
    model       : torch.nn.Module,
    loader      : DataLoader,
    device      : cfg.device,
    precision   : str,
    threshold   : float,
) -> list[dict]:
    """Run batched inference. Returns a list of result dicts.

    Each dict contains:
        image_path   : absolute path to the image
        prediction   : predicted emotion label (or "Unknown" if below threshold)
        confidence   : probability of the predicted class
        Optimistic   : probability score
        Pessimistic  : probability score
        Hostile      : probability score
        Neutral      : probability score
    """
    results    = []

    #allow only BF16 and FP32
    autocast_dtype = torch.bfloat16 if precision == "bf16" else torch.float32

    for images, paths in tqdm(loader, desc = "Inferencing", unit = "batch"):
        images = images.to(device)

        #cast input to model precision
        if precision == "bf16":
            images = images.to(torch.bfloat16)

        with torch.amp.autocast("cuda", dtype = autocast_dtype, enabled = (device.type == "cuda")):
            logits = model(images) #(B, num_classes)

        #always compute probabilities in FP32 for numerical accuracy
        probs = F.softmax(logits.float(), dim = 1).cpu() #(B, num_classes)
        preds = probs.argmax(dim = 1) #(B,)
        confs = probs.max(dim = 1).values #(B,)

        for path, pred, conf, prob in zip(paths, preds, confs, probs):
            pred_idx  = pred.item()
            conf_val  = conf.item()

            #apply confidence threshold, label as Unknown if not confident enough
            if conf_val < cfg.threshold:
                label = "Unknown"
            else:
                label = cfg.EMOTION_NAMES[pred_idx]

            results.append({
                "image_path":  path,
                "prediction":  label,
                "confidence":  round(conf_val, 4),
                **{name: round(prob[i].item(), 4) for i, name in enumerate(cfg.EMOTION_NAMES)},
            })

    return results

In [8]:
def print_summary(results: list[dict], elapsed: float) -> None:
    """Print a summary table to stdout."""
    total    = len(results)
    unknown  = sum(1 for r in results if r["prediction"] == "Unknown")
    counts   = {name: sum(1 for r in results if r["prediction"] == name)
                for name in cfg.EMOTION_NAMES}

    print(f"\n{'-' * 57}")
    print(f"  Results  ({total} images  |  {elapsed:.1f}s  |  "
          f"{total/elapsed:.0f} img/s)")
    print(f"{'-' * 57}")
    
    for name in cfg.EMOTION_NAMES:
        pct = counts[name] / total * 100
        bar = " " * int(pct / 2)
        print(f"  {name:<14} {counts[name]:>5}  ({pct:5.1f}%)  {bar}")
    if unknown:
        pct = unknown / total * 100
        print(f"  {'Unknown':<14} {unknown:>5}  ({pct:5.1f}%)  (below threshold ({cfg.threshold}))")
    print(f"{'-' * 57}\n")

    #show a few example predictions
    print("  Sample predictions:")
    for r in results[:5]:
        name = Path(r["image_path"]).name
        print(f"    {name:<40} → {r['prediction']:<14} ({r['confidence']:.3f})")
    if len(results) > 5:
        print(f"    ... and {len(results) - 5} more\n")

## Inference

In [9]:
#load checkpoint
model, precision = load_model("/home/simon/Documents/Zwischen Wörtern und Pixeln/SWIN-Base_checkpoint_final.safetensors",
           "bf16",
          cfg.device)

Loading model from: /home/simon/Documents/Zwischen Wörtern und Pixeln/SWIN-Base_checkpoint_final.safetensors
  Precision: BF16
  Model: swin_base_patch4_window7_224  |  86.7 M params  |  device: cuda


In [10]:
transforms = get_inference_transform()

In [11]:
from pprint import pprint

#get image paths
images = collect_image_paths("/home/simon/Documents/Zwischen Wörtern und Pixeln/Sample_Dataset/Reichel/media_act")

#verify
pprint(images[:5])
print(len(images))

['/home/simon/Documents/Zwischen Wörtern und '
 'Pixeln/Sample_Dataset/Reichel/media_act/1000272_522859_image.jpeg',
 '/home/simon/Documents/Zwischen Wörtern und '
 'Pixeln/Sample_Dataset/Reichel/media_act/1001168_523479_image.jpeg',
 '/home/simon/Documents/Zwischen Wörtern und '
 'Pixeln/Sample_Dataset/Reichel/media_act/1001169_523480_image.jpeg',
 '/home/simon/Documents/Zwischen Wörtern und '
 'Pixeln/Sample_Dataset/Reichel/media_act/1001175_523486_image.jpeg',
 '/home/simon/Documents/Zwischen Wörtern und '
 'Pixeln/Sample_Dataset/Reichel/media_act/1001183_523496_image.jpeg']
8190


In [12]:
dataset = ImageDataset(images, transforms)

In [13]:
#define dataloader
loader = DataLoader(
    dataset,
    batch_size = cfg.batch_size,
    shuffle = False,
    num_workers = cfg.num_workers,
    pin_memory = True
)

In [14]:
#run inference
start_time = perf_counter()

results = run_inference(
    model = model,
    loader = loader,
    device = cfg.device,
    precision = precision,
    threshold = cfg.threshold
)

end_time = perf_counter()

elapsed = end_time - start_time

Inferencing: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 128/128 [00:10<00:00, 11.65batch/s]


In [15]:
print_summary(results, elapsed)


---------------------------------------------------------
  Results  (8190 images  |  11.0s  |  745 img/s)
---------------------------------------------------------
  Optimistic       870  ( 10.6%)       
  Pessimistic      254  (  3.1%)   
  Hostile          264  (  3.2%)   
  Neutral         2850  ( 34.8%)                   
  Unknown         3952  ( 48.3%)  (below threshold (0.6))
---------------------------------------------------------

  Sample predictions:
    1000272_522859_image.jpeg                → Unknown        (0.575)
    1001168_523479_image.jpeg                → Unknown        (0.383)
    1001169_523480_image.jpeg                → Hostile        (0.871)
    1001175_523486_image.jpeg                → Unknown        (0.452)
    1001183_523496_image.jpeg                → Neutral        (0.790)
    ... and 8185 more



In [16]:
df_results = pd.DataFrame(results)

df_results.head()

,image_path,prediction,confidence,Optimistic,Pessimistic,Hostile,Neutral
0,/home/simon/Documents/Zwischen Wörtern und Pix...,Unknown,0.5746,0.5746,0.1214,0.0251,0.2790
1,/home/simon/Documents/Zwischen Wörtern und Pix...,Unknown,0.3832,0.3832,0.2151,0.0397,0.3621
2,/home/simon/Documents/Zwischen Wörtern und Pix...,Hostile,0.8709,0.0773,0.0017,0.8709,0.0501
3,/home/simon/Documents/Zwischen Wörtern und Pix...,Unknown,0.4518,0.2861,0.0462,0.2159,0.4518
4,/home/simon/Documents/Zwischen Wörtern und Pix...,Neutral,0.7901,0.1174,0.0838,0.0086,0.7901


In [17]:
df_results.to_csv("/home/simon/Documents/Zwischen Wörtern und Pixeln/inference_results_bf16.csv",
                 encoding = "UTF-8",
                 index = False)